In [2]:
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Start Spark
spark = SparkSession.builder \
    .appName("Hypertension_Triage_US12") \
    .getOrCreate()

# 2. Read Parquet
df_spark = spark.read.parquet("analysis_sample.parquet")

# 3. Ensure HIGH_BP_FLAG column exists
if "HIGH_BP_FLAG" not in df_spark.columns:
    df_spark = df_spark.withColumn(
        "HIGH_BP_FLAG",
        F.when((F.col("SYSTOLIC_BP") >= 140) | (F.col("DIASTOLIC_BP") >= 90), 1).otherwise(0)
    )

# 4. Reproduce Week 6 Geopolitical Zone summary using groupBy and agg
zone_summary = (
    df_spark.groupBy("GEOPOLITICAL_ZONE")
    .agg(
        F.count("*").alias("total_screened"),
        F.sum("HIGH_BP_FLAG").alias("hypertensive_cases"),
        F.round(F.avg("SYSTOLIC_BP"), 1).alias("avg_systolic_bp"),
        F.round(F.avg("HIGH_BP_FLAG") * 100, 2).alias("prevalence_pct")
    )
    .orderBy(F.desc("prevalence_pct"))
)

zone_summary.show()

+-----------------+--------------+------------------+---------------+--------------+
|GEOPOLITICAL_ZONE|total_screened|hypertensive_cases|avg_systolic_bp|prevalence_pct|
+-----------------+--------------+------------------+---------------+--------------+
|       South West|         31936|             18688|          124.1|         58.52|
|      South South|         22298|             13039|          123.7|         58.48|
|       North East|         21113|             12233|          123.5|         57.94|
|       South East|         16956|              9815|          123.8|         57.89|
|    North Central|         23549|             13611|          123.6|          57.8|
|       North West|         42042|             24218|          123.5|          57.6|
+-----------------+--------------+------------------+---------------+--------------+



## When Apache Spark Is Worth the Extra Complexity

Apache Spark is worth the extra architectural complexity when dataset sizes exceed single-machine RAM capacity (tens of gigabytes to terabytes) or require parallel processing across multi-node clusters to meet pipeline SLAs. For smaller datasets like our ~158,000-row survey, single-node tools like pandas and pre-aggregated Parquet files are faster and simpler because they avoid JVM startup latency and distributed shuffle overhead.